# Clean and integrate the 2025 procurement data

Combines the twelve monthly files, validates identifiers and dates, preserves source values, and produces the three analysis-ready datasets.

Run this notebook from the `notebooks/` directory after completing the preceding numbered stage. Generated files are written to the documented project directories.


In [ ]:
import json
import os
import glob

ruta_base = "../data/raw/2025"

archivos = glob.glob(os.path.join(ruta_base, "*.json"))

len(archivos)


In [ ]:
todos_los_releases = []

for archivo in archivos:
    with open(archivo, "r", encoding="utf-8") as f:
        paquetes = json.load(f)

    for paquete in paquetes:
        todos_los_releases.extend(paquete["releases"])

len(todos_los_releases)


In [ ]:
ocids = [release["ocid"] for release in todos_los_releases]

len(ocids), len(set(ocids))


In [ ]:
con_awards = sum(
    1 for release in todos_los_releases
    if "awards" in release and release["awards"]
)

sin_awards = len(todos_los_releases) - con_awards

con_awards, sin_awards


In [ ]:
buyers = {
    release["buyer"]["id"]
    for release in todos_los_releases
    if "buyer" in release and release["buyer"].get("id")
}

len(buyers)


In [ ]:
tenderers = set()

for release in todos_los_releases:
    tender = release.get("tender", {})

    for tenderer in tender.get("tenderers", []):
        tenderer_id = tenderer.get("id")
        if tenderer_id:
            tenderers.add(tenderer_id)

len(tenderers)


In [ ]:
suppliers = set()

for release in todos_los_releases:
    for award in release.get("awards", []):
        for supplier in award.get("suppliers", []):
            supplier_id = supplier.get("id")
            if supplier_id:
                suppliers.add(supplier_id)

len(suppliers)


In [ ]:
import pandas as pd

filas_procedimientos = []

for release in todos_los_releases:
    buyer = release.get("buyer", {})
    tender = release.get("tender", {})

    filas_procedimientos.append({
        "ocid": release.get("ocid"),
        "release_date": release.get("date"),
        "buyer_id": buyer.get("id"),
        "buyer_name": buyer.get("name"),
        "tender_id": tender.get("id"),
        "tender_status": tender.get("status"),
        "number_of_tenderers": tender.get("numberOfTenderers")
    })

procedures_df = pd.DataFrame(filas_procedimientos)

procedures_df.shape


In [ ]:
procedures_df.isnull().sum()


In [ ]:
procedures_df[
    procedures_df["number_of_tenderers"].isnull()
].head(10)


In [ ]:
casos_nulos = []

for release in todos_los_releases:
    tender = release.get("tender", {})

    if tender.get("numberOfTenderers") is None:
        casos_nulos.append({
            "ocid": release.get("ocid"),
            "tender_status": tender.get("status"),
            "cantidad_tenderers": len(tender.get("tenderers", []))
        })

pd.DataFrame(casos_nulos)["cantidad_tenderers"].value_counts().sort_index()


In [ ]:
pd.DataFrame(casos_nulos)["tender_status"].value_counts(dropna=False)


In [ ]:
nombres_por_buyer = (
    procedures_df
    .groupby("buyer_id")["buyer_name"]
    .nunique()
)

buyers_con_varios_nombres = nombres_por_buyer[nombres_por_buyer > 1]

len(buyers_con_varios_nombres)


In [ ]:
ejemplos_variaciones_buyer = (
    procedures_df[
        procedures_df["buyer_id"].isin(buyers_con_varios_nombres.index)
    ]
    .groupby("buyer_id")["buyer_name"]
    .unique()
)

ejemplos_variaciones_buyer.head(10)


In [ ]:
import unicodedata
import re

def normalizar_nombre(texto):
    if pd.isna(texto):
        return texto

    texto = str(texto).strip().upper()
    texto = ''.join(
        c for c in unicodedata.normalize("NFD", texto)
        if unicodedata.category(c) != "Mn"
    )
    texto = re.sub(r"\s+", " ", texto)

    return texto

procedures_df["buyer_name_normalized"] = (
    procedures_df["buyer_name"].apply(normalizar_nombre)
)

procedures_df[
    ["buyer_name", "buyer_name_normalized"]
].head()


In [ ]:
nombres_normalizados_por_buyer = (
    procedures_df
    .groupby("buyer_id")["buyer_name_normalized"]
    .nunique()
)

buyers_con_varios_nombres_normalizados = (
    nombres_normalizados_por_buyer[
        nombres_normalizados_por_buyer > 1
    ]
)

len(buyers_con_varios_nombres_normalizados)


In [ ]:
procedures_df["buyer_ruc"] = (
    procedures_df["buyer_id"]
    .str.extract(r"EC-RUC-(\d{13})")
)

ruc_con_varios_ids = (
    procedures_df
    .groupby("buyer_ruc")["buyer_id"]
    .nunique()
)

ruc_con_varios_ids = ruc_con_varios_ids[ruc_con_varios_ids > 1]

len(ruc_con_varios_ids)


In [ ]:
filas_tenderers = []

for release in todos_los_releases:
    ocid = release.get("ocid")
    tender = release.get("tender", {})

    for tenderer in tender.get("tenderers", []):
        filas_tenderers.append({
            "ocid": ocid,
            "tenderer_id": tenderer.get("id"),
            "tenderer_name": tenderer.get("name")
        })

tender_participation_df = pd.DataFrame(filas_tenderers)

tender_participation_df.shape


In [ ]:
# Comprobar si existen participaciones duplicadas:

duplicados_tenderers = tender_participation_df.duplicated(
    subset=["ocid", "tenderer_id"]
).sum()

duplicados_tenderers


In [ ]:
tender_participation_df.isnull().sum()


In [ ]:
filas_awards = []

for release in todos_los_releases:
    ocid = release.get("ocid")

    for award in release.get("awards", []):
        award_id = award.get("id")
        award_date = award.get("date")

        award_value = award.get("value", {})
        award_amount = award_value.get("amount")
        award_currency = award_value.get("currency")

        for supplier in award.get("suppliers", []):
            filas_awards.append({
                "ocid": ocid,
                "award_id": award_id,
                "award_date": award_date,
                "supplier_id": supplier.get("id"),
                "supplier_name": supplier.get("name"),
                "award_amount": award_amount,
                "award_currency": award_currency
            })

awards_df = pd.DataFrame(filas_awards)

awards_df.shape


In [ ]:
awards_df.isnull().sum()


In [ ]:
ocids_award_amount_nulo = awards_df.loc[
    awards_df["award_amount"].isnull(),
    "ocid"
].tolist()

for release in todos_los_releases:
    if release.get("ocid") in ocids_award_amount_nulo:
        print("OCID:", release.get("ocid"))
        print(release.get("awards"))
        print("-" * 80)


In [ ]:
filas_awards = []

for release in todos_los_releases:
    ocid = release.get("ocid")

    for award in release.get("awards", []):
        award_id = award.get("id")
        award_date = award.get("date")

        if award.get("value"):
            monto = award["value"].get("amount")
            moneda = award["value"].get("currency")
            fuente_monto = "award.value"

        elif award.get("correctedValue"):
            monto = award["correctedValue"].get("amount")
            moneda = award["correctedValue"].get("currency")
            fuente_monto = "award.correctedValue"

        else:
            monto = None
            moneda = None
            fuente_monto = None

        for supplier in award.get("suppliers", []):
            filas_awards.append({
                "ocid": ocid,
                "award_id": award_id,
                "award_date": award_date,
                "supplier_id": supplier.get("id"),
                "supplier_name": supplier.get("name"),
                "award_amount": monto,
                "award_currency": moneda,
                "award_amount_source": fuente_monto
            })

awards_df = pd.DataFrame(filas_awards)

awards_df[
    ["award_amount", "award_currency", "award_amount_source"]
].isnull().sum()


In [ ]:
# Contar cuántas adjudicaciones utilizaron award.value

awards_df["award_amount_source"].value_counts()


In [ ]:
# Comprobar si existen montos adjudicados iguales a cero o negativos

montos_cero = (awards_df["award_amount"] == 0).sum()
montos_negativos = (awards_df["award_amount"] < 0).sum()

montos_cero, montos_negativos


In [ ]:
awards_df["award_currency"].value_counts(dropna=False)


In [ ]:
# Comprobar si existen adjudicaciones duplicadas:

duplicados_awards = awards_df.duplicated(
    subset=["ocid", "award_id", "supplier_id"]
).sum()

duplicados_awards


In [ ]:
nombres_por_supplier = (
    awards_df
    .groupby("supplier_id")["supplier_name"]
    .nunique()
)

suppliers_con_varios_nombres = nombres_por_supplier[
    nombres_por_supplier > 1
]

len(suppliers_con_varios_nombres)


In [ ]:
ejemplos_variaciones_supplier = (
    awards_df[
        awards_df["supplier_id"].isin(suppliers_con_varios_nombres.index)
    ]
    .groupby("supplier_id")["supplier_name"]
    .unique()
)

ejemplos_variaciones_supplier.head(10)


In [ ]:
awards_df["supplier_name_normalized"] = (
    awards_df["supplier_name"].apply(normalizar_nombre)
)


nombres_normalizados_por_supplier = (
    awards_df
    .groupby("supplier_id")["supplier_name_normalized"]
    .nunique()
)

suppliers_con_varios_nombres_normalizados = (
    nombres_normalizados_por_supplier[
        nombres_normalizados_por_supplier > 1
    ]
)

len(suppliers_con_varios_nombres_normalizados)


In [ ]:
ejemplos_supplier_persistentes = (
    awards_df[
        awards_df["supplier_id"].isin(
            suppliers_con_varios_nombres_normalizados.index
        )
    ]
    .groupby("supplier_id")["supplier_name"]
    .unique()
)

ejemplos_supplier_persistentes


In [ ]:
awards_df["supplier_ruc"] = (
    awards_df["supplier_id"]
    .str.extract(r"EC-RUC-(\d{13})")
)

ruc_supplier_con_varios_ids = (
    awards_df
    .groupby("supplier_ruc")["supplier_id"]
    .nunique()
)

ruc_supplier_con_varios_ids = (
    ruc_supplier_con_varios_ids[
        ruc_supplier_con_varios_ids > 1
    ]
)

len(ruc_supplier_con_varios_ids)


In [ ]:
suppliers_sin_ruc = awards_df[
    awards_df["supplier_ruc"].isna()
]

print("Filas de adjudicación sin RUC estándar:", len(suppliers_sin_ruc))
print("Proveedores únicos sin RUC estándar:", suppliers_sin_ruc["supplier_id"].nunique())


In [ ]:
proveedores_sin_ruc_unicos = (
    suppliers_sin_ruc[
        ["supplier_id", "supplier_name"]
    ]
    .drop_duplicates()
    .sort_values("supplier_name")
)

proveedores_sin_ruc_unicos.head(30)


In [ ]:
awards_df["supplier_ruc"] = (
    awards_df["supplier_id"]
    .str.extract(r"(?:EC-RUC-|ID-)(\d{13})")
)

print("Filas sin RUC identificable:", awards_df["supplier_ruc"].isna().sum())
print(
    "Proveedores únicos sin RUC identificable:",
    awards_df.loc[
        awards_df["supplier_ruc"].isna(),
        "supplier_id"
    ].nunique()
)


In [ ]:
ruc_con_varios_supplier_ids = (
    awards_df
    .groupby("supplier_ruc")["supplier_id"]
    .nunique()
)

ruc_con_varios_supplier_ids = (
    ruc_con_varios_supplier_ids[
        ruc_con_varios_supplier_ids > 1
    ]
)

len(ruc_con_varios_supplier_ids)


In [ ]:
consorcios_df = awards_df[
    awards_df["supplier_name_normalized"]
    .fillna("")
    .str.contains("CONSORC", case=False)
]

print("Filas de adjudicación asociadas a consorcios:", len(consorcios_df))
print("Consorcios únicos por RUC:", consorcios_df["supplier_ruc"].nunique())


In [ ]:
consorcios_unicos = (
    consorcios_df[
        ["supplier_ruc", "supplier_id", "supplier_name"]
    ]
    .drop_duplicates()
    .sort_values("supplier_name")
)

consorcios_unicos


In [ ]:
rucs_consorcios = set(consorcios_df["supplier_ruc"].dropna())

parties_consorcios = []

for release in todos_los_releases:
    for party in release.get("parties", []):
        party_id = party.get("id", "")

        ruc_encontrado = None
        match = re.search(r"(?:EC-RUC-|ID-)(\d{13})", str(party_id))
        if match:
            ruc_encontrado = match.group(1)

        if ruc_encontrado in rucs_consorcios:
            parties_consorcios.append(party)

len(parties_consorcios)


In [ ]:
campos_consorcios = sorted(
    set().union(*(party.keys() for party in parties_consorcios))
)

campos_consorcios


In [ ]:
# Esto permitirá documentar si existen adjudicaciones anteriores o posteriores a 2025.

awards_df["award_date_dt"] = pd.to_datetime(
    awards_df["award_date"],
    utc=True
)

awards_df["award_date_dt"].min(), awards_df["award_date_dt"].max()


In [ ]:
awards_df["award_year"] = awards_df["award_date_dt"].dt.year

awards_df["award_year"].value_counts().sort_index()


In [ ]:
procedures_df["release_date_dt"] = pd.to_datetime(
    procedures_df["release_date"],
    utc=True
)

procedures_df["release_year"] = procedures_df["release_date_dt"].dt.year

procedures_df["release_year"].value_counts().sort_index()


In [ ]:
fechas_inicio_tender = []

for release in todos_los_releases:
    tender = release.get("tender", {})
    tender_period = tender.get("tenderPeriod", {})

    fechas_inicio_tender.append({
        "ocid": release.get("ocid"),
        "tender_start_date": tender_period.get("startDate")
    })

tender_dates_df = pd.DataFrame(fechas_inicio_tender)

tender_dates_df["tender_start_date_dt"] = pd.to_datetime(
    tender_dates_df["tender_start_date"],
    utc=True,
    errors="coerce"
)

print("Fechas de inicio nulas:", tender_dates_df["tender_start_date_dt"].isna().sum())

tender_dates_df["tender_start_date_dt"].dt.year.value_counts().sort_index()


In [ ]:
ocids_inicio_2026 = tender_dates_df.loc[
    tender_dates_df["tender_start_date_dt"].dt.year == 2026,
    "ocid"
]

procedimientos_inicio_2026 = procedures_df[
    procedures_df["ocid"].isin(ocids_inicio_2026)
][
    ["ocid", "release_date", "buyer_name", "tender_id", "tender_status"]
]

procedimientos_inicio_2026


In [ ]:
# pero actualizados o iniciados formalmente en 2026.

revision_27 = tender_dates_df[
    tender_dates_df["ocid"].isin(ocids_inicio_2026)
].merge(
    procedures_df[
        ["ocid", "tender_id", "release_date", "tender_status"]
    ],
    on="ocid",
    how="left"
)

revision_27[
    ["ocid", "tender_id", "tender_start_date", "release_date", "tender_status"]
]


In [ ]:
tender_dates_df["tender_start_year_local"] = (
    tender_dates_df["tender_start_date"]
    .str[:4]
    .astype(int)
)

tender_dates_df["tender_start_year_local"].value_counts().sort_index()


In [ ]:
awards_df["award_year_local"] = (
    awards_df["award_date"]
    .str[:4]
    .astype(int)
)

awards_df["award_year_local"].value_counts().sort_index()


In [ ]:
revision_valor_tender = []

for release in todos_los_releases:
    tender = release.get("tender", {})
    lots = tender.get("lots", [])

    revision_valor_tender.append({
        "ocid": release.get("ocid"),
        "tiene_tender_value": tender.get("value") is not None,
        "numero_lotes": len(lots)
    })

revision_valor_tender_df = pd.DataFrame(revision_valor_tender)

print(
    "Procedimientos con tender.value:",
    revision_valor_tender_df["tiene_tender_value"].sum()
)

print(
    "Distribución del número de lotes:"
)

revision_valor_tender_df["numero_lotes"].value_counts().sort_index()


In [ ]:
procedimiento_sin_lote = revision_valor_tender_df[
    revision_valor_tender_df["numero_lotes"] == 0
]

procedimiento_sin_lote


In [ ]:
ocid_sin_lote = "ocds-5wno2w-SIE-CNELM-017A-2011-124705"

for release in todos_los_releases:
    if release.get("ocid") == ocid_sin_lote:
        print("OCID:", release.get("ocid"))
        print("Fecha release:", release.get("date"))
        print("Tender:")
        print(release.get("tender"))
        print("\nPlanning:")
        print(release.get("planning"))
        print("\nAwards:")
        print(release.get("awards"))
        break


In [ ]:
procedures_df["buyer_ruc_valido"] = (
    procedures_df["buyer_id"]
    .str.extract(r"^EC-RUC-(\d{13})-")[0]
)

print(
    "Procedimientos con buyer_id sin RUC válido de 13 dígitos:",
    procedures_df["buyer_ruc_valido"].isna().sum()
)

procedures_df[
    procedures_df["buyer_ruc_valido"].isna()
][
    ["ocid", "buyer_id", "buyer_name", "tender_id", "tender_status"]
]


In [ ]:
awards_df["supplier_ruc_validado"] = (
    awards_df["supplier_id"]
    .str.extract(r"^(?:EC-RUC-|ID-)(\d{13})-")[0]
)

print(
    "Adjudicaciones con supplier_id sin RUC válido de 13 dígitos:",
    awards_df["supplier_ruc_validado"].isna().sum()
)

awards_df[
    awards_df["supplier_ruc_validado"].isna()
][
    ["ocid", "supplier_id", "supplier_name"]
]


In [ ]:
supplier_ids_atipicos = set(
    awards_df.loc[
        awards_df["supplier_ruc_validado"].isna(),
        "supplier_id"
    ]
)

datos_parties_atipicos = []

for release in todos_los_releases:
    for party in release.get("parties", []):
        if party.get("id") in supplier_ids_atipicos:
            identifier = party.get("identifier", {})

            datos_parties_atipicos.append({
                "party_id": party.get("id"),
                "party_name": party.get("name"),
                "identifier_id": identifier.get("id"),
                "identifier_scheme": identifier.get("scheme"),
                "legal_name": identifier.get("legalName")
            })

pd.DataFrame(datos_parties_atipicos).drop_duplicates()


In [ ]:
valores_referenciales = []

for release in todos_los_releases:
    tender = release.get("tender", {})
    lots = tender.get("lots", [])

    if len(lots) > 0:
        value = lots[0].get("value", {})
        tender_value = value.get("amount")
        tender_currency = value.get("currency")
        tender_value_source = "tender.lots[0].value"
    else:
        tender_value = None
        tender_currency = None
        tender_value_source = None

    valores_referenciales.append({
        "ocid": release.get("ocid"),
        "tender_value": tender_value,
        "tender_currency": tender_currency,
        "tender_value_source": tender_value_source
    })

tender_values_df = pd.DataFrame(valores_referenciales)

tender_values_df.isnull().sum()


In [ ]:
tender_values_df["tender_currency"].value_counts(dropna=False)


In [ ]:
# Comprobar si existen valores referenciales iguales a cero o negativos.

tender_valores_cero = (tender_values_df["tender_value"] == 0).sum()
tender_valores_negativos = (tender_values_df["tender_value"] < 0).sum()

tender_valores_cero, tender_valores_negativos


In [ ]:
cantidad_items_award = []

for release in todos_los_releases:
    for award in release.get("awards", []):
        cantidad_items_award.append(
            len(award.get("items", []))
        )

pd.Series(cantidad_items_award).value_counts().sort_index()


In [ ]:
filas_cpc = []

for release in todos_los_releases:
    ocid = release.get("ocid")

    for award in release.get("awards", []):
        items = award.get("items", [])

        if items:
            clasificacion = items[0].get("classification", {})

            cpc_id = clasificacion.get("id")
            cpc_scheme = clasificacion.get("scheme")
            cpc_description = clasificacion.get("description")
        else:
            cpc_id = None
            cpc_scheme = None
            cpc_description = None

        filas_cpc.append({
            "ocid": ocid,
            "award_id": award.get("id"),
            "cpc_id": cpc_id,
            "cpc_scheme": cpc_scheme,
            "cpc_description": cpc_description
        })

cpc_df = pd.DataFrame(filas_cpc)

cpc_df.isnull().sum()


In [ ]:
cpc_df["cpc_5"] = (
    cpc_df["cpc_id"]
    .astype(str)
    .str.replace(r"\D", "", regex=True)
    .str[:5]
)


cpc_df[
    ["cpc_id", "cpc_5", "cpc_description"]
].head(10)


In [ ]:
cpc_df["longitud_cpc_5"] = cpc_df["cpc_5"].str.len()

cpc_df["longitud_cpc_5"].value_counts().sort_index()


In [ ]:
awards_df = awards_df.merge(
    cpc_df[
        ["ocid", "award_id", "cpc_id", "cpc_5", "cpc_description"]
    ],
    on=["ocid", "award_id"],
    how="left"
)

awards_df[
    ["ocid", "supplier_name", "cpc_id", "cpc_5"]
].head()


In [ ]:
awards_df.shape


In [ ]:
awards_df = awards_df.merge(
    procedures_df[
        ["ocid", "buyer_id", "buyer_name", "buyer_name_normalized"]
    ],
    on="ocid",
    how="left"
)

awards_df[
    ["ocid", "buyer_name", "supplier_name", "cpc_5", "award_date"]
].head()


In [ ]:
awards_df.shape


In [ ]:
awards_df[
    ["buyer_id", "supplier_id", "cpc_5", "award_date"]
].isnull().sum()


In [ ]:
modalidades = []

for release in todos_los_releases:
    tender = release.get("tender", {})

    modalidades.append({
        "procurement_method": tender.get("procurementMethod"),
        "procurement_method_details": tender.get("procurementMethodDetails")
    })

modalidades_df = pd.DataFrame(modalidades)

modalidades_df["procurement_method_details"].value_counts(dropna=False)


In [ ]:
awards_df["award_date_local"] = pd.to_datetime(
    awards_df["award_date"].str[:19],
    errors="coerce"
)

print(
    "Fechas de adjudicación nulas después de la conversión:",
    awards_df["award_date_local"].isna().sum()
)

print(
    "Fecha mínima:",
    awards_df["award_date_local"].min()
)

print(
    "Fecha máxima:",
    awards_df["award_date_local"].max()
)


In [ ]:
awards_recurrencia_df = awards_df.sort_values(
    by=[
        "buyer_id",
        "supplier_id",
        "cpc_5",
        "award_date_local"
    ]
).copy()

awards_recurrencia_df[
    [
        "buyer_name",
        "supplier_name",
        "cpc_5",
        "award_date_local"
    ]
].head(10)


In [ ]:
#

resultados_recurrencia = []

for (buyer_id, supplier_id, cpc_5), grupo in awards_recurrencia_df.groupby(
    ["buyer_id", "supplier_id", "cpc_5"]
):

    grupo = grupo.sort_values("award_date_local").copy()
    fechas = grupo["award_date_local"].tolist()
    ocids = grupo["ocid"].tolist()

    max_procesos_90d = 0
    fecha_inicio_max = None
    fecha_fin_max = None

    inicio = 0

    for fin in range(len(fechas)):

        while fechas[fin] - fechas[inicio] > pd.Timedelta(days=90):
            inicio += 1

        cantidad = len(set(ocids[inicio:fin + 1]))

        if cantidad > max_procesos_90d:
            max_procesos_90d = cantidad
            fecha_inicio_max = fechas[inicio]
            fecha_fin_max = fechas[fin]

    resultados_recurrencia.append({
        "buyer_id": buyer_id,
        "supplier_id": supplier_id,
        "cpc_5": cpc_5,
        "max_procesos_90d": max_procesos_90d,
        "fecha_inicio_ventana": fecha_inicio_max,
        "fecha_fin_ventana": fecha_fin_max
    })

recurrencia_90d_df = pd.DataFrame(resultados_recurrencia)

casos_recurrentes_90d = recurrencia_90d_df[
    recurrencia_90d_df["max_procesos_90d"] > 3
]

len(casos_recurrentes_90d)


In [ ]:
buyer_names = (
    awards_df[
        ["buyer_id", "buyer_name"]
    ]
    .drop_duplicates(subset=["buyer_id"])
)

supplier_names = (
    awards_df[
        ["supplier_id", "supplier_name"]
    ]
    .drop_duplicates(subset=["supplier_id"])
)

casos_recurrentes_detalle = (
    casos_recurrentes_90d
    .merge(buyer_names, on="buyer_id", how="left")
    .merge(supplier_names, on="supplier_id", how="left")
    .sort_values(
        "max_procesos_90d",
        ascending=False
    )
)

casos_recurrentes_detalle[
    [
        "buyer_name",
        "supplier_name",
        "cpc_5",
        "max_procesos_90d",
        "fecha_inicio_ventana",
        "fecha_fin_ventana"
    ]
]


In [ ]:
casos_recurrentes_detalle[
    "max_procesos_90d"
].value_counts().sort_index()


In [ ]:
estados_award = []

for release in todos_los_releases:
    for award in release.get("awards", []):
        estados_award.append({
            "ocid": release.get("ocid"),
            "award_id": award.get("id"),
            "award_status": award.get("status")
        })

award_status_df = pd.DataFrame(estados_award)

print(
    "Adjudicaciones con award.status nulo:",
    award_status_df["award_status"].isna().sum()
)

award_status_df["award_status"].value_counts(dropna=False)


In [ ]:
fechas_fin_tender = []

for release in todos_los_releases:
    tender = release.get("tender", {})
    tender_period = tender.get("tenderPeriod", {})

    fechas_fin_tender.append({
        "ocid": release.get("ocid"),
        "tender_end_date": tender_period.get("endDate")
    })

tender_end_df = pd.DataFrame(fechas_fin_tender)

print(
    "Procedimientos con tenderPeriod.endDate nulo:",
    tender_end_df["tender_end_date"].isna().sum()
)

tender_end_df["tender_end_date"].isna().value_counts()


In [ ]:
comparacion_oferentes = []

for release in todos_los_releases:
    tender = release.get("tender", {})

    numero_reportado = tender.get("numberOfTenderers")
    cantidad_observada = len(tender.get("tenderers", []))

    comparacion_oferentes.append({
        "ocid": release.get("ocid"),
        "numero_reportado": numero_reportado,
        "cantidad_observada": cantidad_observada
    })

comparacion_oferentes_df = pd.DataFrame(comparacion_oferentes)

casos_comparables = comparacion_oferentes_df[
    comparacion_oferentes_df["numero_reportado"].notna()
]

diferencias_oferentes = casos_comparables[
    casos_comparables["numero_reportado"] !=
    casos_comparables["cantidad_observada"]
]

print("Procedimientos comparables:", len(casos_comparables))
print("Procedimientos con diferencias:", len(diferencias_oferentes))


In [ ]:
Q1 = awards_df["award_amount"].quantile(0.25)
Q3 = awards_df["award_amount"].quantile(0.75)

IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

outliers_award = awards_df[
    (awards_df["award_amount"] < limite_inferior) |
    (awards_df["award_amount"] > limite_superior)
]

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Límite superior:", limite_superior)
print("Valores extremos identificados:", len(outliers_award))


In [ ]:
porcentaje_outliers = len(outliers_award) / len(awards_df) * 100
monto_maximo = awards_df["award_amount"].max()

print("Porcentaje de valores extremos:", round(porcentaje_outliers, 2), "%")
print("Monto adjudicado máximo:", monto_maximo)


In [ ]:
percentil_99 = awards_df["award_amount"].quantile(0.99)

casos_sobre_p99 = (
    awards_df["award_amount"] > percentil_99
).sum()

print("Percentil 99:", percentil_99)
print("Adjudicaciones por encima del percentil 99:", casos_sobre_p99)


In [ ]:
import numpy as np

awards_df["award_amount_log10"] = np.log10(
    awards_df["award_amount"]
)

awards_df["award_amount_winsor_p99"] = (
    awards_df["award_amount"].clip(upper=percentil_99)
)

awards_df[
    [
        "award_amount",
        "award_amount_log10",
        "award_amount_winsor_p99"
    ]
].describe()


In [ ]:
# Detectar nombres registrados como texto "null", "none" o vacío.

def contar_textos_nulos(serie):
    return (
        serie.astype(str)
        .str.strip()
        .str.lower()
        .isin(["null", "none", ""])
        .sum()
    )

print(
    "Buyer names problemáticos:",
    contar_textos_nulos(procedures_df["buyer_name"])
)

print(
    "Tenderer names problemáticos:",
    contar_textos_nulos(tender_participation_df["tenderer_name"])
)

print(
    "Supplier names problemáticos:",
    contar_textos_nulos(awards_df["supplier_name"])
)


In [ ]:
# texto nulo, vacío o "none".

valores_nulos_texto = ["null", "none", ""]

oferentes_nombre_problematico = tender_participation_df[
    tender_participation_df["tenderer_name"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(valores_nulos_texto)
][["ocid", "tenderer_id", "tenderer_name"]]

proveedores_nombre_problematico = awards_df[
    awards_df["supplier_name"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(valores_nulos_texto)
][["ocid", "supplier_id", "supplier_name"]]

print("Oferentes problemáticos:")
display(oferentes_nombre_problematico)

print("Proveedores adjudicados problemáticos:")
display(proveedores_nombre_problematico)


In [ ]:
ids_oferentes_problematicos = set(
    oferentes_nombre_problematico["tenderer_id"]
)

nombres_parties_oferentes = []

for release in todos_los_releases:
    for party in release.get("parties", []):
        if party.get("id") in ids_oferentes_problematicos:
            nombre = party.get("name")

            nombres_parties_oferentes.append({
                "tenderer_id": party.get("id"),
                "nombre_parties": nombre
            })

nombres_parties_oferentes_df = (
    pd.DataFrame(nombres_parties_oferentes)
    .drop_duplicates()
)

print(
    "IDs problemáticos únicos:",
    oferentes_nombre_problematico["tenderer_id"].nunique()
)

nombres_parties_oferentes_df


In [ ]:
nombres_parties_validos = nombres_parties_oferentes_df[
    ~nombres_parties_oferentes_df["nombre_parties"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["null", "none", ""])
].copy()

nombres_parties_validos["nombre_normalizado"] = (
    nombres_parties_validos["nombre_parties"]
    .apply(normalizar_nombre)
)

resumen_nombres_oferentes = (
    nombres_parties_validos
    .groupby("tenderer_id")
    .agg(
        nombres_validos=("nombre_parties", "count"),
        nombres_normalizados_distintos=("nombre_normalizado", "nunique")
    )
)

print("IDs problemáticos únicos:", oferentes_nombre_problematico["tenderer_id"].nunique())
print("IDs con al menos un nombre válido en parties:", resumen_nombres_oferentes.shape[0])

resumen_nombres_oferentes


In [ ]:
nombres_validos_por_oferente = (
    nombres_parties_validos
    .groupby("tenderer_id")["nombre_parties"]
    .unique()
)

nombres_validos_por_oferente


In [ ]:
ids_oferentes_problematicos = set(
    oferentes_nombre_problematico["tenderer_id"]
)

nombres_validos_misma_tabla = tender_participation_df[
    tender_participation_df["tenderer_id"].isin(ids_oferentes_problematicos)
].copy()

nombres_validos_misma_tabla = nombres_validos_misma_tabla[
    ~nombres_validos_misma_tabla["tenderer_name"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["null", "none", ""])
]

nombres_validos_misma_tabla.groupby(
    "tenderer_id"
)["tenderer_name"].unique()


In [ ]:
ids_con_nombre_valido = set(
    nombres_validos_misma_tabla["tenderer_id"]
)

ids_problematicos_totales = set(
    oferentes_nombre_problematico["tenderer_id"]
)

ids_sin_nombre_recuperable = (
    ids_problematicos_totales - ids_con_nombre_valido
)

print("IDs sin nombre recuperable:", len(ids_sin_nombre_recuperable))

oferentes_sin_nombre_recuperable = (
    oferentes_nombre_problematico[
        oferentes_nombre_problematico["tenderer_id"].isin(
            ids_sin_nombre_recuperable
        )
    ]
)

print(
    "Registros afectados:",
    len(oferentes_sin_nombre_recuperable)
)

oferentes_sin_nombre_recuperable[
    ["tenderer_id", "tenderer_name"]
].drop_duplicates()


In [ ]:
nombres_validos_por_id = (
    tender_participation_df[
        ~tender_participation_df["tenderer_name"]
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(["null", "none", ""])
    ]
    .groupby("tenderer_id")["tenderer_name"]
    .agg(lambda x: x.value_counts().index[0])
    .to_dict()
)

tender_participation_df["nombre_oferente_limpio"] = (
    tender_participation_df["tenderer_name"]
)

mascara_nombre_problematico = (
    tender_participation_df["tenderer_name"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["null", "none", ""])
)

tender_participation_df.loc[
    mascara_nombre_problematico,
    "nombre_oferente_limpio"
] = tender_participation_df.loc[
    mascara_nombre_problematico,
    "tenderer_id"
].map(nombres_validos_por_id)

print(
    "Registros originalmente problemáticos:",
    mascara_nombre_problematico.sum()
)

print(
    "Registros recuperados:",
    tender_participation_df.loc[
        mascara_nombre_problematico,
        "nombre_oferente_limpio"
    ].notna().sum()
)

print(
    "Registros sin nombre recuperable:",
    tender_participation_df.loc[
        mascara_nombre_problematico,
        "nombre_oferente_limpio"
    ].isna().sum()
)


In [ ]:
ids_proveedores_problematicos = set(
    proveedores_nombre_problematico["supplier_id"]
)

nombres_validos_proveedores = awards_df[
    awards_df["supplier_id"].isin(ids_proveedores_problematicos)
].copy()

nombres_validos_proveedores = nombres_validos_proveedores[
    ~nombres_validos_proveedores["supplier_name"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["null", "none", ""])
]

print(
    "IDs problemáticos únicos:",
    proveedores_nombre_problematico["supplier_id"].nunique()
)

print(
    "IDs con al menos un nombre válido:",
    nombres_validos_proveedores["supplier_id"].nunique()
)

nombres_validos_proveedores.groupby(
    "supplier_id"
)["supplier_name"].unique()


In [ ]:
ids_proveedores_con_nombre = set(
    nombres_validos_proveedores["supplier_id"]
)

ids_proveedores_problematicos_totales = set(
    proveedores_nombre_problematico["supplier_id"]
)

ids_proveedores_sin_nombre = (
    ids_proveedores_problematicos_totales - ids_proveedores_con_nombre
)

print(
    "IDs de proveedores sin nombre recuperable:",
    len(ids_proveedores_sin_nombre)
)

proveedores_sin_nombre_recuperable = (
    proveedores_nombre_problematico[
        proveedores_nombre_problematico["supplier_id"].isin(
            ids_proveedores_sin_nombre
        )
    ]
)

print(
    "Registros afectados:",
    len(proveedores_sin_nombre_recuperable)
)

proveedores_sin_nombre_recuperable[
    ["supplier_id", "supplier_name"]
].drop_duplicates()


In [ ]:
# evitando inferencias o emparejamientos aproximados.

nombres_parties_proveedores = []

for release in todos_los_releases:
    for party in release.get("parties", []):
        if party.get("id") in ids_proveedores_sin_nombre:

            nombre = party.get("name")
            identificador = party.get("identifier", {})

            nombres_parties_proveedores.append({
                "supplier_id": party.get("id"),
                "nombre_parties": nombre,
                "nombre_legal": identificador.get("legalName")
            })

nombres_parties_proveedores_df = (
    pd.DataFrame(nombres_parties_proveedores)
    .drop_duplicates()
)

nombres_parties_proveedores_df


In [ ]:
#
# Prioridad:
# 1. Nombre válido observado en otras adjudicaciones.
# 2. Nombre válido encontrado en parties.
#

awards_df["nombre_proveedor_limpio"] = awards_df["supplier_name"]

mascara_proveedor_problematico = (
    awards_df["supplier_name"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["null", "none", ""])
)

nombres_validos_awards = (
    awards_df[
        ~awards_df["supplier_name"]
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(["null", "none", ""])
    ]
    .groupby("supplier_id")["supplier_name"]
    .agg(lambda x: x.value_counts().index[0])
    .to_dict()
)

parties_validos = nombres_parties_proveedores_df[
    ~nombres_parties_proveedores_df["nombre_parties"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["null", "none", ""])
]

nombres_validos_parties = (
    parties_validos
    .drop_duplicates("supplier_id")
    .set_index("supplier_id")["nombre_parties"]
    .to_dict()
)

# Primero intentar recuperar desde otras adjudicaciones
awards_df.loc[
    mascara_proveedor_problematico,
    "nombre_proveedor_limpio"
] = awards_df.loc[
    mascara_proveedor_problematico,
    "supplier_id"
].map(nombres_validos_awards)

mascara_aun_sin_nombre = (
    mascara_proveedor_problematico &
    awards_df["nombre_proveedor_limpio"].isna()
)

awards_df.loc[
    mascara_aun_sin_nombre,
    "nombre_proveedor_limpio"
] = awards_df.loc[
    mascara_aun_sin_nombre,
    "supplier_id"
].map(nombres_validos_parties)

print(
    "Registros originalmente problemáticos:",
    mascara_proveedor_problematico.sum()
)

print(
    "Registros recuperados:",
    awards_df.loc[
        mascara_proveedor_problematico,
        "nombre_proveedor_limpio"
    ].notna().sum()
)

print(
    "Registros sin nombre recuperable:",
    awards_df.loc[
        mascara_proveedor_problematico,
        "nombre_proveedor_limpio"
    ].isna().sum()
)

print(
    "IDs únicos sin nombre recuperable:",
    awards_df.loc[
        mascara_proveedor_problematico &
        awards_df["nombre_proveedor_limpio"].isna(),
        "supplier_id"
    ].nunique()
)


In [ ]:
proveedores_finales_sin_nombre = (
    awards_df.loc[
        mascara_proveedor_problematico &
        awards_df["nombre_proveedor_limpio"].isna(),
        ["supplier_id", "supplier_ruc", "ocid"]
    ]
    .drop_duplicates()
)

proveedores_finales_sin_nombre


In [ ]:
awards_df["nombre_proveedor_normalizado"] = (
    awards_df["nombre_proveedor_limpio"]
    .apply(normalizar_nombre)
)

print(
    "Nombres normalizados nulos:",
    awards_df["nombre_proveedor_normalizado"].isna().sum()
)

awards_df[
    [
        "supplier_id",
        "supplier_name",
        "nombre_proveedor_limpio",
        "nombre_proveedor_normalizado"
    ]
].head()


In [ ]:
awards_df["nombre_proveedor_normalizado"].isna().sum()


In [ ]:
tender_participation_df["nombre_oferente_normalizado"] = (
    tender_participation_df["nombre_oferente_limpio"]
    .apply(normalizar_nombre)
)

print(
    "Nombres de oferentes normalizados nulos:",
    tender_participation_df["nombre_oferente_normalizado"].isna().sum()
)


In [ ]:
tender_participation_df["fuente_nombre_oferente"] = "original"

tender_participation_df.loc[
    mascara_nombre_problematico &
    tender_participation_df["nombre_oferente_limpio"].notna(),
    "fuente_nombre_oferente"
] = "recuperado_mismo_id"

tender_participation_df.loc[
    tender_participation_df["nombre_oferente_limpio"].isna(),
    "fuente_nombre_oferente"
] = "no_informado"

tender_participation_df["fuente_nombre_oferente"].value_counts()


In [ ]:
awards_df["fuente_nombre_proveedor"] = "original"

awards_df.loc[
    mascara_proveedor_problematico &
    awards_df["nombre_proveedor_limpio"].notna(),
    "fuente_nombre_proveedor"
] = "recuperado_mismo_id"

awards_df.loc[
    awards_df["nombre_proveedor_limpio"].isna(),
    "fuente_nombre_proveedor"
] = "no_informado"

awards_df["fuente_nombre_proveedor"].value_counts()


In [ ]:
print("=== PROCEDIMIENTOS ===")
print("Filas:", len(procedures_df))
print("OCID duplicados:", procedures_df["ocid"].duplicated().sum())
print("OCID nulos:", procedures_df["ocid"].isna().sum())

print("\n=== PARTICIPACIÓN DE OFERENTES ===")
print("Filas:", len(tender_participation_df))
print(
    "Duplicados OCID + oferente:",
    tender_participation_df.duplicated(
        subset=["ocid", "tenderer_id"]
    ).sum()
)
print("IDs de oferente nulos:", tender_participation_df["tenderer_id"].isna().sum())

print("\n=== ADJUDICACIONES ===")
print("Filas:", len(awards_df))
print(
    "Duplicados OCID + adjudicación + proveedor:",
    awards_df.duplicated(
        subset=["ocid", "award_id", "supplier_id"]
    ).sum()
)
print("IDs de proveedor nulos:", awards_df["supplier_id"].isna().sum())
print("Montos adjudicados nulos:", awards_df["award_amount"].isna().sum())
print("CPC5 nulos:", awards_df["cpc_5"].isna().sum())


In [ ]:
print("=== PROCEDIMIENTOS ===")
print("Filas:", len(procedures_df))
print("OCID duplicados:", procedures_df["ocid"].duplicated().sum())
print("OCID nulos:", procedures_df["ocid"].isna().sum())


In [ ]:
# Integrar valor referencial
procedures_df = procedures_df.merge(
    tender_values_df[
        ["ocid", "tender_value", "tender_currency", "tender_value_source"]
    ],
    on="ocid",
    how="left"
)

procedures_df = procedures_df.merge(
    tender_dates_df[
        ["ocid", "tender_start_date", "tender_start_year_local"]
    ],
    on="ocid",
    how="left"
)

ocids_con_adjudicacion = set(awards_df["ocid"])

procedures_df["tiene_adjudicacion"] = (
    procedures_df["ocid"].isin(ocids_con_adjudicacion)
)

procedures_df.shape


In [ ]:
tender_participation_df["ruc_oferente_validado"] = (
    tender_participation_df["tenderer_id"]
    .str.extract(r"^(?:EC-RUC-|ID-)(\d{13})-")[0]
)

print(
    "Participaciones con tenderer_id sin RUC válido de 13 dígitos:",
    tender_participation_df["ruc_oferente_validado"].isna().sum()
)

print(
    "Oferentes únicos con identificador atípico:",
    tender_participation_df.loc[
        tender_participation_df["ruc_oferente_validado"].isna(),
        "tenderer_id"
    ].nunique()
)

tender_participation_df[
    tender_participation_df["ruc_oferente_validado"].isna()
][
    ["tenderer_id", "tenderer_name"]
].drop_duplicates()


In [ ]:
ids_oferentes_atipicos = set(
    tender_participation_df.loc[
        tender_participation_df["ruc_oferente_validado"].isna(),
        "tenderer_id"
    ]
)

datos_oferentes_atipicos = []

for release in todos_los_releases:
    for party in release.get("parties", []):
        if party.get("id") in ids_oferentes_atipicos:

            identificador = party.get("identifier", {})

            datos_oferentes_atipicos.append({
                "tenderer_id": party.get("id"),
                "nombre": party.get("name"),
                "identifier_id": identificador.get("id"),
                "identifier_scheme": identificador.get("scheme"),
                "nombre_legal": identificador.get("legalName")
            })

oferentes_atipicos_parties_df = (
    pd.DataFrame(datos_oferentes_atipicos)
    .drop_duplicates()
)

oferentes_atipicos_parties_df


In [ ]:
todos_ids_oferentes = set(
    tender_participation_df["tenderer_id"].dropna().unique()
)

revision_ruc_atipicos = []

for tenderer_id in ids_oferentes_atipicos:

    coincidencia = re.match(
        r"^EC-RUC-(\d+)-",
        str(tenderer_id)
    )

    if coincidencia:
        numero = coincidencia.group(1)

        if len(numero) == 14:
            posible_ruc_13 = numero[:13]

            coincidencias_13 = [
                actor_id
                for actor_id in todos_ids_oferentes
                if f"EC-RUC-{posible_ruc_13}-" in actor_id
            ]

            revision_ruc_atipicos.append({
                "tenderer_id_atipico": tenderer_id,
                "numero_digitos": len(numero),
                "posible_ruc_13": posible_ruc_13,
                "coincidencias_en_base": coincidencias_13
            })

revision_ruc_atipicos_df = pd.DataFrame(revision_ruc_atipicos)

revision_ruc_atipicos_df


In [ ]:
nombre_busqueda = "RIOVISTA ENTERPRISE SAS"

coincidencias_riovista = []

for release in todos_los_releases:
    for party in release.get("parties", []):
        nombre = str(party.get("name", "")).strip().upper()

        if nombre == nombre_busqueda:
            identificador = party.get("identifier", {})

            coincidencias_riovista.append({
                "party_id": party.get("id"),
                "nombre": party.get("name"),
                "identifier_id": identificador.get("id"),
                "identifier_scheme": identificador.get("scheme")
            })

pd.DataFrame(coincidencias_riovista).drop_duplicates()


In [ ]:
tender_dates_df["fecha_inicio_local"] = pd.to_datetime(
    tender_dates_df["tender_start_date"].str[:19],
    errors="coerce"
)

awards_temporal_df = awards_df.merge(
    tender_dates_df[
        ["ocid", "fecha_inicio_local"]
    ],
    on="ocid",
    how="left"
)

adjudicaciones_antes_inicio = awards_temporal_df[
    awards_temporal_df["award_date_local"] <
    awards_temporal_df["fecha_inicio_local"]
]

print(
    "Adjudicaciones con fecha anterior al inicio del procedimiento:",
    len(adjudicaciones_antes_inicio)
)

adjudicaciones_antes_inicio[
    [
        "ocid",
        "fecha_inicio_local",
        "award_date_local",
        "buyer_name",
        "supplier_id"
    ]
].head(20)


In [ ]:
# Esta comprobación permite detectar posibles discrepancias institucionales

comparacion_comprador = []

for release in todos_los_releases:
    buyer = release.get("buyer", {})
    tender = release.get("tender", {})
    procuring_entity = tender.get("procuringEntity", {})

    comparacion_comprador.append({
        "ocid": release.get("ocid"),
        "buyer_id": buyer.get("id"),
        "buyer_name": buyer.get("name"),
        "procuring_entity_id": procuring_entity.get("id"),
        "procuring_entity_name": procuring_entity.get("name")
    })

comparacion_comprador_df = pd.DataFrame(comparacion_comprador)

print(
    "ProcuringEntity ID nulos:",
    comparacion_comprador_df["procuring_entity_id"].isna().sum()
)

diferencias_comprador = comparacion_comprador_df[
    comparacion_comprador_df["buyer_id"] !=
    comparacion_comprador_df["procuring_entity_id"]
]

print(
    "Procedimientos con buyer.id distinto de procuringEntity.id:",
    len(diferencias_comprador)
)

diferencias_comprador.head(20)


In [ ]:
perfil_numerico = pd.DataFrame({
    "numero_oferentes": procedures_df["number_of_tenderers"],
    "valor_referencial": procedures_df["tender_value"],
})


print("=== PROCEDIMIENTOS ===")
display(
    perfil_numerico.describe().T
)

print("\nValores nulos:")
display(
    perfil_numerico.isna().sum()
)

print("\n=== ADJUDICACIONES ===")
display(
    awards_df[["award_amount"]].describe().T
)

print("\nMontos adjudicados nulos:")
print(
    awards_df["award_amount"].isna().sum()
)


In [ ]:
ocid_registro_prueba = "ocds-5wno2w-SIE-CNELM-017A-2011-124705"

procedures_df["registro_atipico_calidad"] = (
    procedures_df["ocid"] == ocid_registro_prueba
)

print(
    "Registros marcados como atípicos:",
    procedures_df["registro_atipico_calidad"].sum()
)

procedures_df.loc[
    procedures_df["registro_atipico_calidad"],
    [
        "ocid",
        "buyer_id",
        "buyer_name",
        "tender_id",
        "tender_status",
        "tender_value",
        "tiene_adjudicacion"
    ]
]


In [ ]:
# Entidades contratantes
procedures_df["identificador_comprador_atipico"] = (
    procedures_df["buyer_ruc_valido"].isna()
)

# Oferentes
tender_participation_df["identificador_oferente_atipico"] = (
    tender_participation_df["ruc_oferente_validado"].isna()
)

# Proveedores adjudicados
awards_df["identificador_proveedor_atipico"] = (
    awards_df["supplier_ruc_validado"].isna()
)

print(
    "Procedimientos con identificador de comprador atípico:",
    procedures_df["identificador_comprador_atipico"].sum()
)

print(
    "Participaciones con identificador de oferente atípico:",
    tender_participation_df["identificador_oferente_atipico"].sum()
)

print(
    "Adjudicaciones con identificador de proveedor atípico:",
    awards_df["identificador_proveedor_atipico"].sum()
)


In [ ]:
#

ruta_procesados = "../data/processed"

os.makedirs(ruta_procesados, exist_ok=True)

procedures_df.to_csv(
    os.path.join(
        ruta_procesados,
        "procedures_2025.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

tender_participation_df.to_csv(
    os.path.join(
        ruta_procesados,
        "tender_participation_2025.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

awards_df.to_csv(
    os.path.join(
        ruta_procesados,
        "awards_2025.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

print("Bases procesadas guardadas correctamente.")
